In [1]:
import numpy as np
from nvidia import nvcomp
import cupy as cp

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "6"

# import urllib.request
# urllib.request.urlretrieve("http://textfiles.com/etext/NONFICTION/locke-essay-113.txt", "locke-essay-113.txt")
# urllib.request.urlretrieve("http://textfiles.com/etext/FICTION/mobydick.txt", "mobydick.txt")

print("nvcomp version:", nvcomp.__version__)
print("nvcomp cuda version:", nvcomp.__cuda_version__)

# bit = 2.96
bit = 4
len = bit * 4096 * 4096 // 8
ascending = np.arange(0, len, dtype=np.uint8)
# ascending = np.random.randint(4096, 4096, dtype=np.int32)
nvarr_h = nvcomp.as_array(ascending)

print(ascending.__array_interface__)
print(nvarr_h.__array_interface__)
print(nvarr_h.__cuda_array_interface__)
print(nvarr_h.buffer_size)
print(nvarr_h.buffer_kind)
print(nvarr_h.ndim)
print(nvarr_h.dtype)
print(nvarr_h.shape)
print(nvarr_h.strides)
print(nvarr_h.item_size)
print(nvarr_h.size)

data_gpu = cp.array(ascending)
nvarr_d = nvcomp.as_array(data_gpu)
print(data_gpu.__cuda_array_interface__)
print(nvarr_d.__cuda_array_interface__)
print(nvarr_d.buffer_kind)
print(nvarr_d.ndim)
print(nvarr_d.dtype)
print(nvarr_d.shape)
print(nvarr_d.strides)
print(nvarr_d.item_size)
print(nvarr_d.size)

nvcomp version: 5.0.0
nvcomp cuda version: 12090
{'data': (139936137158672, False), 'strides': None, 'descr': [('', '|u1')], 'typestr': '|u1', 'shape': (8388608,), 'version': 3}
{'shape': (8388608,), 'strides': None, 'typestr': '|u1', 'data': (139936137158672, False), 'version': 3}
{'shape': (8388608,), 'strides': None, 'typestr': '|u1', 'data': (139936137158672, False), 'version': 3, 'stream': 1}
8388608
ArrayBufferKind.STRIDED_HOST
1
uint8
(8388608,)
(1,)
1
8388608
{'shape': (8388608,), 'typestr': '|u1', 'descr': [('', '|u1')], 'stream': 1, 'version': 3, 'strides': None, 'data': (139935409504256, False)}
{'shape': (8388608,), 'strides': None, 'typestr': '|u1', 'data': (139935409504256, False), 'version': 3, 'stream': 1}
ArrayBufferKind.STRIDED_DEVICE
1
uint8
(8388608,)
(1,)
1
8388608


In [ ]:
nvarr_d_cnv = nvarr_h.cuda()
print(nvarr_d_cnv.__cuda_array_interface__)

nvarr_h_cnv = nvarr_d.cpu()
print(nvarr_h_cnv.__array_interface__)

ans_codec = nvcomp.Codec(algorithm="ANS", chunk_size=262144, checksum_policy = nvcomp.ChecksumPolicy.COMPUTE_AND_VERIFY)
# ans_codec = nvcomp.Codec(algorithm="ANS", checksum_policy = nvcomp.ChecksumPolicy.COMPUTE_AND_VERIFY)
ans_comp_arr = ans_codec.encode(nvarr_d_cnv)

print(ans_comp_arr.buffer_size)
# print(ans_codec.__cuda_array_interface__)
# print(ans_codec.buffer_kind)

ans_deco_arr_uint8 = ans_codec.decode(ans_comp_arr)
ans_deco_arr_uint32 = ans_codec.decode(ans_comp_arr, '<u4')

print(ans_deco_arr_uint8.dtype)
print(ans_deco_arr_uint32.dtype)

{'shape': (8388608,), 'strides': None, 'typestr': '|u1', 'data': (13009233920, False), 'version': 3, 'stream': 1}
{'shape': (8388608,), 'strides': None, 'typestr': '|u1', 'data': (140260176560128, False), 'version': 3}
8790088
int8
uint32


In [ ]:
print("Uncompressed size is", nvarr_d_cnv.buffer_size)
# algos = ["LZ4", "Snappy", "Bitcomp", "ANS", "Zstd",  "Cascaded"]
algos = ["LZ4", "Snappy", "Bitcomp", "ANS"]
bitstreams = [
    nvcomp.BitstreamKind.NVCOMP_NATIVE,
    nvcomp.BitstreamKind.RAW,
    nvcomp.BitstreamKind.WITH_UNCOMPRESSED_SIZE
]

for algorithm in algos:
    for bitstream_kind in bitstreams:
        codec = nvcomp.Codec(algorithm=algorithm, bitstream_kind=bitstream_kind)
        comp_arr = codec.encode(nvarr_d_cnv)
        comp_ratio = comp_arr.buffer_size/nvarr_d_cnv.buffer_size
        print("Compressed size for", algorithm, "with bitstream", bitstream_kind, "is", comp_arr.buffer_size, "({:.1%})".format(comp_ratio))
        decomp_array = codec.decode(comp_arr)
        print ("is equal to original? -", bytes(decomp_array.cpu()) ==  bytes(nvarr_d_cnv.cpu()))

Uncompressed size is 8388608
Compressed size for LZ4 with bitstream BitstreamKind.NVCOMP_NATIVE is 69698 (0.8%)
is equal to original? - True
Compressed size for LZ4 with bitstream BitstreamKind.RAW is 33162 (0.4%)
is equal to original? - True
Compressed size for LZ4 with bitstream BitstreamKind.WITH_UNCOMPRESSED_SIZE is 33166 (0.4%)
is equal to original? - True
Compressed size for Snappy with bitstream BitstreamKind.NVCOMP_NATIVE is 2636868 (31.4%)
is equal to original? - True
Compressed size for Snappy with bitstream BitstreamKind.RAW is 2601096 (31.0%)
is equal to original? - True
Compressed size for Snappy with bitstream BitstreamKind.WITH_UNCOMPRESSED_SIZE is 2601104 (31.0%)
is equal to original? - True
Compressed size for Bitcomp with bitstream BitstreamKind.NVCOMP_NATIVE is 2586696 (30.8%)
is equal to original? - True
Compressed size for Bitcomp with bitstream BitstreamKind.RAW is 2580512 (30.8%)
is equal to original? - True
Compressed size for Bitcomp with bitstream BitstreamKin

In [ ]:
print("Uncompressed size is", nvarr_d_cnv.buffer_size)

algos = ["ANS"] # 사용자가 지정한 알고리즘
bitstreams = [
    nvcomp.BitstreamKind.NVCOMP_NATIVE,
    nvcomp.BitstreamKind.RAW,
    nvcomp.BitstreamKind.WITH_UNCOMPRESSED_SIZE
]

# 통계 측정을 위한 설정 (이벤트 객체 생성 부하를 고려해 1000회로 설정 권장)
num_repeats = 100 

for algorithm in algos:
    for bitstream_kind in bitstreams:
        try:
            codec = nvcomp.Codec(algorithm=algorithm, bitstream_kind=bitstream_kind, chunk_size=262144*32)
            comp_arr = codec.encode(nvarr_d_cnv)
            comp_ratio = comp_arr.buffer_size/nvarr_d_cnv.buffer_size
            
            print("Compressed size for", algorithm, "with bitstream", bitstream_kind, "is", comp_arr.buffer_size, "({:.1%})".format(comp_ratio))
            
            # --- [추가됨] Latency 측정 (Mean & Std) ---
            # 1. 웜업
            _ = codec.decode(comp_arr)
            cp.cuda.Stream.null.synchronize()

            # 2. 개별 시간 측정을 위한 이벤트 리스트 생성
            start_events = [cp.cuda.Event() for _ in range(num_repeats)]
            end_events = [cp.cuda.Event() for _ in range(num_repeats)]

            # 3. 반복 실행 및 기록
            for i in range(num_repeats):
                start_events[i].record()
                decomp_array = codec.decode(comp_arr)
                end_events[i].record()
            
            end_events[-1].synchronize()

            # 4. 통계 계산
            latencies = [cp.cuda.get_elapsed_time(s, e) for s, e in zip(start_events, end_events)]
            mean_latency = np.mean(latencies)
            std_latency = np.std(latencies)

            print(f"Latency: {mean_latency:.4f} ms ± {std_latency:.4f} ms")
            # ------------------------------------------

            print ("is equal to original? -", bytes(decomp_array.cpu()) ==  bytes(nvarr_d_cnv.cpu()))
            print("-" * 50)

        except Exception as e:
             print(f"Skipping {algorithm} with {bitstream_kind}: {e}")

Uncompressed size is 8388608
Compressed size for ANS with bitstream BitstreamKind.NVCOMP_NATIVE is 8789064 (104.8%)
Latency: 0.0954 ms ± 0.0083 ms
is equal to original? - True
--------------------------------------------------
Compressed size for ANS with bitstream BitstreamKind.RAW is 8394280 (100.1%)
Latency: 0.6421 ms ± 0.1614 ms
is equal to original? - True
--------------------------------------------------
Compressed size for ANS with bitstream BitstreamKind.WITH_UNCOMPRESSED_SIZE is 8394288 (100.1%)
Latency: 0.6333 ms ± 0.0145 ms
is equal to original? - True
--------------------------------------------------


Uncompressed size is 8388608
Compressed size for ANS with bitstream BitstreamKind.NVCOMP_NATIVE is 8789064 (104.8%)
Latency: 0.0980 ms ± 0.0212 ms
is equal to original? - True
--------------------------------------------------
Compressed size for ANS with bitstream BitstreamKind.RAW is 8394280 (100.1%)
Latency: 0.6391 ms ± 0.0273 ms
is equal to original? - True
--------------------------------------------------
Compressed size for ANS with bitstream BitstreamKind.WITH_UNCOMPRESSED_SIZE is 8394288 (100.1%)
Latency: 0.6317 ms ± 0.0136 ms
is equal to original? - True
--------------------------------------------------